In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import re

file_path = "/content/drive/MyDrive/NLP_final_project/news_sentiment_notebook5.parquet"
df = pd.read_parquet(file_path, engine="pyarrow")

print(df.shape)
print(df.columns.tolist())
df.head()

(129326, 24)
['url', 'date', 'title', 'text', 'full_text', 'relevance_score', 'ai_core_hits', 'impact_hits', 'business_hits', 'industry_hits', 'industry_text', 'industries', 'industry_count', 'org_entities_raw', 'org_entities', 'org_count', 'theme_text', 'impact_themes', 'theme_count', 'text_for_sentiment', 'predicted_sentiment', 'sent_prob_negative', 'sent_prob_neutral', 'sent_prob_positive']


,url,date,title,text,full_text,relevance_score,ai_core_hits,impact_hits,business_hits,industry_hits,...,org_entities,org_count,theme_text,impact_themes,theme_count,text_for_sentiment,predicted_sentiment,sent_prob_negative,sent_prob_neutral,sent_prob_positive
0,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,4,1,0,1,0,...,"[Wonderful Products (Contact Support, Werners ...",3,this ai video of gymnastics might be the freak...,"[Decision Support / Analytics, Cybersecurity /...",3,This AI video of gymnastics might be the freak...,neutral,0.304103,0.399602,0.296296
1,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,"If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...",19,6,0,0,1,...,"[Wonderful Products (Contact Support, SEO]",2,if using ai feels like a chore try this boing ...,"[Augmentation / Assistance, Decision Support /...",3,"If using AI feels like a chore, try this - Boi...",positive,0.307933,0.329882,0.362184
2,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,19,4,3,0,1,...,[AI Foundation Model is Shaping the Future of ...,7,the road ahead how china s ai foundation model...,[Decision Support / Analytics],1,The Road Ahead: How China's AI Foundation Mode...,neutral,0.273275,0.406151,0.320574
3,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,11,3,1,0,0,...,"[Microsoft, өтуСенбі, Қала, Windows, Windows A...",9,microsoft and nvidia to empower developers wit...,[Productivity / Efficiency],1,Microsoft and Nvidia to Empower Developers wit...,positive,0.217208,0.295443,0.487349
4,https://citylife.capetown/lb/uncategorized/how...,2023-12-12,Google Releases New Chatbot Bard as a Strong C...,Google Releases New Chatbot Bard as a Strong C...,Google Releases New Chatbot Bard as a Strong C...,18,6,0,0,0,...,"[Google, OpenAI, Wiessel un InhaltDi, Stad Lie...",8,google releases new chatbot bard as a strong c...,"[Customer Service / Personalization, Software ...",2,Google Releases New Chatbot Bard as a Strong C...,positive,0.226581,0.219392,0.554027


In [3]:
import re

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["factor_text"] = df["full_text"].apply(normalize_text)

In [4]:
SUCCESS_DICT = {
    "Data Availability / Quality": [
        "high quality data", "data quality", "clean data", "proprietary data",
        "labeled data", "data availability", "training data"
    ],
    "Leadership / Strategy": [
        "leadership support", "executive support", "top management",
        "business strategy", "strategic priority", "clear strategy"
    ],
    "Talent / Skills": [
        "ai talent", "data scientists", "engineers", "upskilling",
        "training employees", "technical expertise", "skilled workforce"
    ],
    "Infrastructure / Compute": [
        "cloud infrastructure", "compute capacity", "gpu", "data center",
        "scalable infrastructure", "platform investment"
    ],
    "Workflow Integration": [
        "integrate into workflow", "workflow integration", "embedded in workflow",
        "operational integration", "business process integration"
    ],
    "User Adoption / Trust": [
        "user adoption", "employee adoption", "trust in ai",
        "human in the loop", "user acceptance", "confidence in the system"
    ],
    "Cost Savings / ROI": [
        "roi", "return on investment", "cost savings", "reduce costs",
        "business value", "economic benefit"
    ],
    "Regulatory Clarity / Governance": [
        "clear regulation", "governance framework", "compliance readiness",
        "responsible ai framework", "risk management framework"
    ],
    "Partnerships / Ecosystem": [
        "partnership", "partnerships", "ecosystem", "vendor support",
        "technology partner", "collaboration"
    ]
}

In [5]:
BARRIER_DICT = {
    "Poor Data Quality / Access": [
        "poor data quality", "lack of data", "insufficient data",
        "fragmented data", "data silos", "dirty data"
    ],
    "Talent Shortage": [
        "talent shortage", "skills gap", "lack of expertise",
        "shortage of ai talent", "hard to hire", "not enough engineers"
    ],
    "High Cost / Unclear ROI": [
        "high cost", "expensive", "unclear roi", "uncertain roi",
        "costly", "capital intensive", "budget constraints"
    ],
    "Integration Difficulty": [
        "integration challenges", "hard to integrate", "legacy systems",
        "workflow disruption", "implementation complexity"
    ],
    "Trust / Accuracy Concerns": [
        "hallucination", "accuracy concerns", "unreliable", "lack of trust",
        "false positives", "false negatives", "model error"
    ],
    "Regulation / Legal Risk": [
        "regulatory risk", "legal risk", "compliance concerns",
        "liability", "copyright concerns", "privacy concerns"
    ],
    "Security / Privacy Risk": [
        "security risk", "privacy risk", "data breach", "cyber risk",
        "confidentiality concerns", "sensitive data"
    ],
    "Bias / Ethical Concerns": [
        "bias", "fairness", "ethical concerns", "discrimination",
        "responsible ai concerns"
    ],
    "Resistance to Change": [
        "resistance to change", "employee resistance", "organizational resistance",
        "adoption resistance", "change management"
    ],
    "Job Loss / Workforce Fear": [
        "job loss", "layoffs", "displacement", "fear of replacement",
        "replace workers", "workforce anxiety"
    ]
}

In [6]:
SUCCESS_DICT_LOWER = {
    k: [x.lower() for x in v] for k, v in SUCCESS_DICT.items()
}
BARRIER_DICT_LOWER = {
    k: [x.lower() for x in v] for k, v in BARRIER_DICT.items()
}

def get_matching_labels_fast(text, label_dict):
    matches = []
    for label, keywords in label_dict.items():
        for kw in keywords:
            if kw in text:
                matches.append(label)
                break
    return matches

In [7]:
df["success_factors"] = df["factor_text"].apply(
    lambda x: get_matching_labels_fast(x, SUCCESS_DICT_LOWER)
)

df["barrier_factors"] = df["factor_text"].apply(
    lambda x: get_matching_labels_fast(x, BARRIER_DICT_LOWER)
)

df["success_count"] = df["success_factors"].apply(len)
df["barrier_count"] = df["barrier_factors"].apply(len)

df[["title", "success_factors", "barrier_factors"]].head(10)

,title,success_factors,barrier_factors
0,This AI video of gymnastics might be the freak...,[],[]
1,"If using AI feels like a chore, try this - Boi...",[],[]
2,The Road Ahead: How China's AI Foundation Mode...,[],[]
3,Microsoft and Nvidia to Empower Developers wit...,[Infrastructure / Compute],[Regulation / Legal Risk]
4,Google Releases New Chatbot Bard as a Strong C...,[],[Trust / Accuracy Concerns]
5,Zoom Expands AI Offering with AI Companion and...,[],[]
6,Pro-AI Thinking: Enhancing Industrial Environm...,[Partnerships / Ecosystem],[]
7,Best AI Prompts for Business Risk Management,[Partnerships / Ecosystem],[]
8,Bullfrog AI Holdings Inc. [BFRG] Revenue clock...,[],[]
9,IIM Ahmedabad student writes project using Cha...,[],[]


In [8]:
print("No success-factor articles:", (df["success_count"] == 0).sum())
print("Pct no success factors:", round((df["success_count"] == 0).mean() * 100, 2), "%")

print("No barrier-factor articles:", (df["barrier_count"] == 0).sum())
print("Pct no barrier factors:", round((df["barrier_count"] == 0).mean() * 100, 2), "%")

No success-factor articles: 57893
Pct no success factors: 44.77 %
No barrier-factor articles: 90070
Pct no barrier factors: 69.65 %


In [9]:
df_success = df.explode("success_factors").copy()
df_success = df_success[df_success["success_factors"].notna()]

success_summary = (
    df_success.groupby("success_factors")
    .size()
    .reset_index(name="article_count")
    .sort_values("article_count", ascending=False)
)

success_summary

,success_factors,article_count
4,Partnerships / Ecosystem,43131
0,Cost Savings / ROI,22150
2,Infrastructure / Compute,17723
6,Talent / Skills,11207
1,Data Availability / Quality,6839
3,Leadership / Strategy,1452
7,User Adoption / Trust,1407
5,Regulatory Clarity / Governance,1006
8,Workflow Integration,139


In [10]:
df_barrier = df.explode("barrier_factors").copy()
df_barrier = df_barrier[df_barrier["barrier_factors"].notna()]

barrier_summary = (
    df_barrier.groupby("barrier_factors")
    .size()
    .reset_index(name="article_count")
    .sort_values("article_count", ascending=False)
)

barrier_summary

,barrier_factors,article_count
0,Bias / Ethical Concerns,13402
5,Regulation / Legal Risk,12265
1,High Cost / Unclear ROI,7767
7,Security / Privacy Risk,5444
3,Job Loss / Workforce Fear,4822
9,Trust / Accuracy Concerns,4077
8,Talent Shortage,999
4,Poor Data Quality / Access,671
2,Integration Difficulty,656
6,Resistance to Change,531


In [11]:
df_success_ind = df.copy().explode("industries").explode("success_factors")
df_success_ind = df_success_ind[
    df_success_ind["industries"].notna() &
    df_success_ind["success_factors"].notna()
].copy()

success_by_industry = (
    df_success_ind.groupby(["industries", "success_factors"])
    .size()
    .reset_index(name="article_count")
    .sort_values(["industries", "article_count"], ascending=[True, False])
)

success_by_industry.head(30)

,industries,success_factors,article_count
4,Consulting & Professional Services,Partnerships / Ecosystem,6973
0,Consulting & Professional Services,Cost Savings / ROI,3057
2,Consulting & Professional Services,Infrastructure / Compute,2175
6,Consulting & Professional Services,Talent / Skills,2085
1,Consulting & Professional Services,Data Availability / Quality,1223
3,Consulting & Professional Services,Leadership / Strategy,506
7,Consulting & Professional Services,User Adoption / Trust,336
5,Consulting & Professional Services,Regulatory Clarity / Governance,270
8,Consulting & Professional Services,Workflow Integration,53
13,Education,Partnerships / Ecosystem,20371


In [12]:
df_barrier_ind = df.copy().explode("industries").explode("barrier_factors")
df_barrier_ind = df_barrier_ind[
    df_barrier_ind["industries"].notna() &
    df_barrier_ind["barrier_factors"].notna()
].copy()

barrier_by_industry = (
    df_barrier_ind.groupby(["industries", "barrier_factors"])
    .size()
    .reset_index(name="article_count")
    .sort_values(["industries", "article_count"], ascending=[True, False])
)

barrier_by_industry.head(30)

,industries,barrier_factors,article_count
0,Consulting & Professional Services,Bias / Ethical Concerns,2464
5,Consulting & Professional Services,Regulation / Legal Risk,2250
1,Consulting & Professional Services,High Cost / Unclear ROI,1227
7,Consulting & Professional Services,Security / Privacy Risk,1084
3,Consulting & Professional Services,Job Loss / Workforce Fear,763
9,Consulting & Professional Services,Trust / Accuracy Concerns,689
8,Consulting & Professional Services,Talent Shortage,285
6,Consulting & Professional Services,Resistance to Change,222
2,Consulting & Professional Services,Integration Difficulty,200
4,Consulting & Professional Services,Poor Data Quality / Access,146


In [13]:
top_success_by_industry = (
    success_by_industry.groupby("industries", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top_barriers_by_industry = (
    barrier_by_industry.groupby("industries", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top_success_by_industry.head(40)

,industries,success_factors,article_count
0,Consulting & Professional Services,Partnerships / Ecosystem,6973
1,Consulting & Professional Services,Cost Savings / ROI,3057
2,Consulting & Professional Services,Infrastructure / Compute,2175
3,Consulting & Professional Services,Talent / Skills,2085
4,Consulting & Professional Services,Data Availability / Quality,1223
5,Education,Partnerships / Ecosystem,20371
6,Education,Cost Savings / ROI,10138
7,Education,Infrastructure / Compute,7265
8,Education,Talent / Skills,5683
9,Education,Data Availability / Quality,3264


In [14]:
top_barriers_by_industry.head(40)

,industries,barrier_factors,article_count
0,Consulting & Professional Services,Bias / Ethical Concerns,2464
1,Consulting & Professional Services,Regulation / Legal Risk,2250
2,Consulting & Professional Services,High Cost / Unclear ROI,1227
3,Consulting & Professional Services,Security / Privacy Risk,1084
4,Consulting & Professional Services,Job Loss / Workforce Fear,763
5,Education,Bias / Ethical Concerns,7612
6,Education,Regulation / Legal Risk,5870
7,Education,High Cost / Unclear ROI,3777
8,Education,Security / Privacy Risk,2512
9,Education,Job Loss / Workforce Fear,2466


In [15]:
df_barrier_sent = df.copy().explode("barrier_factors")
df_barrier_sent = df_barrier_sent[df_barrier_sent["barrier_factors"].notna()].copy()

barrier_by_sentiment = (
    df_barrier_sent.groupby(["barrier_factors", "predicted_sentiment"])
    .size()
    .reset_index(name="article_count")
)

barrier_by_sentiment.head(30)

,barrier_factors,predicted_sentiment,article_count
0,Bias / Ethical Concerns,negative,4985
1,Bias / Ethical Concerns,neutral,4407
2,Bias / Ethical Concerns,positive,4010
3,High Cost / Unclear ROI,negative,1879
4,High Cost / Unclear ROI,neutral,2718
5,High Cost / Unclear ROI,positive,3170
6,Integration Difficulty,negative,193
7,Integration Difficulty,neutral,168
8,Integration Difficulty,positive,295
9,Job Loss / Workforce Fear,negative,1747


In [16]:
df.to_parquet(
    "/content/drive/MyDrive/NLP_final_project/news_adoption_factors_notebook6.parquet",
    engine="pyarrow",
    index=False
)

In [17]:
success_summary.to_csv(
    "/content/drive/MyDrive/NLP_final_project/success_summary_notebook6.csv",
    index=False
)

barrier_summary.to_csv(
    "/content/drive/MyDrive/NLP_final_project/barrier_summary_notebook6.csv",
    index=False
)

success_by_industry.to_csv(
    "/content/drive/MyDrive/NLP_final_project/success_by_industry_notebook6.csv",
    index=False
)

barrier_by_industry.to_csv(
    "/content/drive/MyDrive/NLP_final_project/barrier_by_industry_notebook6.csv",
    index=False
)

top_success_by_industry.to_csv(
    "/content/drive/MyDrive/NLP_final_project/top_success_by_industry_notebook6.csv",
    index=False
)

top_barriers_by_industry.to_csv(
    "/content/drive/MyDrive/NLP_final_project/top_barriers_by_industry_notebook6.csv",
    index=False
)

print("Saved Notebook 6 outputs.")

Saved Notebook 6 outputs.
